## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q diffusers transformers accelerate safetensors
!pip install -q invisible-watermark>=0.2.0
!pip install -q xformers
!pip install -q Pillow

# Install headless OpenCV (preferred for Colab)
!pip install -q opencv-python-headless

# Install segmentation packages for automatic person masking
!pip install -q segment-anything
!pip install -q groundingdino-py

# Install facenet-pytorch without dependencies to avoid strict version conflicts
!pip install -q --no-deps facenet-pytorch

# Install R2/S3 and environment packages
!pip install -q boto3 python-dotenv requests

# Download SAM model checkpoint
import os
SAM_CHECKPOINT = "/content/sam_vit_h_4b8939.pth"
if not os.path.exists(SAM_CHECKPOINT):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -O {SAM_CHECKPOINT}
    print("SAM model downloaded!")
else:
    print("SAM model already exists.")

In [ ]:
import torch
import gc
import os
import json
import requests
import cv2
import boto3
from io import BytesIO
from pathlib import Path
from PIL import Image, ImageFilter
import numpy as np

from diffusers import (
    StableDiffusionXLInpaintPipeline,
    DPMSolverMultistepScheduler,
)
from diffusers.utils import load_image, make_image_grid

# Segmentation imports
from segment_anything import sam_model_registry, SamPredictor
from transformers import pipeline, AutoProcessor, AutoModelForZeroShotObjectDetection

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Modify these settings as needed
# =============================================================================

# Model settings
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
REFINER_MODEL = "stabilityai/stable-diffusion-xl-refiner-1.0"

# LoRA settings - Update this path to your LoRA file
# Upload your LoRA to Colab or use a HuggingFace path
LORA_PATH = "/content/drive/MyDrive/Loras/wesleygram/output/wesleygram-25.safetensors"
LORA_SCALE = 0.95  # Reduced to preserve source identity better

# Inference settings
RESOLUTION = 1024  # Output resolution (1024x1024)
NUM_INFERENCE_STEPS = 40  # Increased steps for higher guidance

# --- STRATEGY CONFIGURATION ---

# [TEMPLATES]
# These templates wait for the dynamic {pose} to be injected by the Smart Inpainting cell.

# 1. HEADSHOT / FACE REPLACEMENT (Default)
PROMPT_HEADSHOT_TEMPLATE = "(wesley_kamau:1.1) person male, black man, dark skin tone, {pose}, professional headshot, photorealistic, realistic lighting, soft skin texture, same body, same clothing, same pose, natural lighting"
STRENGTH_HEADSHOT = 0.65  # Reduced strength to keep source features
GUIDANCE_HEADSHOT = 8.0   # Slightly reduced guidance for more natural blend

# 2. FULL BODY REPLACEMENT
PROMPT_FULLBODY_TEMPLATE = "(wesley_kamau:1.1) person male, black man, {pose}, full body photo, realistic body proportions, natural pose, photorealistic, consistent lighting, soft skin texture"
STRENGTH_FULLBODY = 0.70  # Reduced strength
GUIDANCE_FULLBODY = 8.0

# Defaults (Fallbacks if smart logic is skipped)
PROMPT = PROMPT_HEADSHOT_TEMPLATE.format(pose="looking at camera")
INPAINT_STRENGTH = STRENGTH_HEADSHOT
GUIDANCE_SCALE = GUIDANCE_HEADSHOT

# Universal Negative Prompt
NEGATIVE_PROMPT = "white skin, pale skin, caucasian features, distorted face, wrong identity, extra limbs, plastic skin, deformed anatomy, uncanny, blurry, low quality"

# Refiner settings
HIGH_NOISE_FRAC = 0.95

# Seed for reproducibility (-1 for random)
SEED = -1

# Person segmentation settings
SAM_CHECKPOINT = "/content/sam_vit_h_4b8939.pth"
SAM_MODEL_TYPE = "vit_h"
DETECTION_MODEL = "IDEA-Research/grounding-dino-tiny"

# Masking Strategy Defaults
PERSON_DETECTION_THRESHOLD = 0.3
MASK_EXPANSION_PIXELS = 10  # 5-10% expansion for hair blending
MASK_FEATHER_RADIUS = 10  # Soft edges for realism

# R2 Storage settings
R2_BUCKET_NAME = "instagram-profiles"

# Device and dtype
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Device: {DEVICE}")
print(f"Dtype: {DTYPE}")

## 3. R2 Credentials Setup

Configure R2 credentials using either:
- **Google Colab**: Secrets (Settings → Secrets → Add new secret)
- **Local**: `.env` file with `python-dotenv`

Required secrets/env vars:
- `R2_ENDPOINT_URL` - Your R2 endpoint
- `R2_ACCESS_KEY_ID` - Access key
- `R2_SECRET_ACCESS_KEY` - Secret key
- `R2_BUCKET_NAME` (optional, defaults to `instagram-profiles`)

In [ ]:
# =============================================================================
# R2 CREDENTIALS SETUP
# Supports both Google Colab secrets and local .env files
# =============================================================================

def load_credentials():
    """
    Load R2 credentials from Colab secrets or .env file.

    Returns:
        dict with r2_endpoint, r2_access_key, r2_secret_key, r2_bucket
    """
    credentials = {}

    # Try Google Colab secrets first
    try:
        from google.colab import userdata
        credentials['r2_endpoint'] = userdata.get('R2_ENDPOINT')
        credentials['r2_access_key'] = userdata.get('R2_ACCESS_KEY')
        credentials['r2_secret_key'] = userdata.get('R2_SECRET_KEY')
        credentials['r2_bucket'] = userdata.get('R2_BUCKET') or R2_BUCKET_NAME
        print("✓ Loaded credentials:", {k: "***" for k in credentials})
        print("✓ Loaded credentials from Colab secrets")
    except (ImportError, Exception) as e:
        print(f"Colab secrets not available: {e}")

        # Fall back to .env file for local development
        try:
            from dotenv import load_dotenv
            load_dotenv()
            credentials['r2_endpoint'] = os.getenv('R2_ENDPOINT_URL') or os.getenv('R2_ENDPOINT')
            credentials['r2_access_key'] = os.getenv('R2_ACCESS_KEY_ID') or os.getenv('R2_ACCESS_KEY')
            credentials['r2_secret_key'] = os.getenv('R2_SECRET_ACCESS_KEY') or os.getenv('R2_SECRET_KEY')
            credentials['r2_bucket'] = os.getenv('R2_BUCKET') or os.getenv('R2_BUCKET_NAME') or R2_BUCKET_NAME
            print("✓ Loaded credentials from .env file:", {k: "***" for k in credentials})
        except ImportError:
            print("⚠ python-dotenv not installed, using environment variables only")
            credentials['r2_endpoint'] = os.getenv('R2_ENDPOINT_URL')
            credentials['r2_access_key'] = os.getenv('R2_ACCESS_KEY_ID')
            credentials['r2_secret_key'] = os.getenv('R2_SECRET_ACCESS_KEY')
            credentials['r2_bucket'] = os.getenv('R2_BUCKET_NAME') or R2_BUCKET_NAME

    # Validate credentials
    if not all([credentials.get('r2_endpoint'), credentials.get('r2_access_key'), credentials.get('r2_secret_key')]):
        missing = [k for k, v in credentials.items() if not v and k != 'r2_bucket']
        raise ValueError(f"Missing R2 credentials: {missing}\n"
                        "Please set them in Colab secrets or .env file.")

    return credentials

# Load credentials
r2_creds = load_credentials()
print(f"R2 Endpoint: {r2_creds['r2_endpoint']}")
print(f"R2 Bucket: {r2_creds['r2_bucket']}")

In [ ]:
# =============================================================================
# R2 CLIENT AND PROFILE FETCHING
# =============================================================================

class R2ProfileFetcher:
    """
    Fetches Instagram profile photos from R2 storage.
    Uses profiles_metadata.json to map usernames to R2 keys.
    """

    def __init__(self, credentials: dict, metadata_path: str = None):
        """
        Initialize R2 client and load metadata.

        Args:
            credentials: Dict with r2_endpoint, r2_access_key, r2_secret_key, r2_bucket
            metadata_path: Local path to profiles_metadata.json (optional)
        """
        self.bucket = credentials['r2_bucket']

        # Initialize S3 client for R2
        self.s3_client = boto3.client(
            's3',
            endpoint_url=credentials['r2_endpoint'],
            aws_access_key_id=credentials['r2_access_key'],
            aws_secret_access_key=credentials['r2_secret_key']
        )

        # Load profiles metadata
        self.metadata = self._load_metadata(metadata_path)
        self.profiles = self.metadata.get('profiles', {})

        # Build username lookup index
        self.username_to_id = {
            p['username'].lower(): pid
            for pid, p in self.profiles.items()
            if 'username' in p
        }

        print(f"✓ Loaded {len(self.profiles)} profiles from metadata")

    def _load_metadata(self, local_path: str = None) -> dict:
        """Load profiles_metadata.json from local file or R2."""

        # Try local file first
        if local_path and os.path.exists(local_path):
            print(f"Loading metadata from local file: {local_path}")
            with open(local_path, 'r', encoding='utf-8') as f:
                return json.load(f)

        # Try to fetch from R2
        try:
            print("Fetching metadata from R2...")
            response = self.s3_client.get_object(
                Bucket=self.bucket,
                Key='profiles_metadata.json'
            )
            return json.loads(response['Body'].read().decode('utf-8'))
        except Exception as e:
            print(f"Could not fetch from R2: {e}")

        # Try Colab upload as fallback
        try:
            from google.colab import files
            print("\nPlease upload profiles_metadata.json:")
            uploaded = files.upload()
            if uploaded:
                filename = list(uploaded.keys())[0]
                with open(f"/content/{filename}", 'r', encoding='utf-8') as f:
                    return json.load(f)
        except ImportError:
            pass

        raise FileNotFoundError("Could not load profiles_metadata.json")

    def get_profile_by_username(self, username: str) -> dict:
        """
        Get profile data by username.

        Args:
            username: Instagram username (case-insensitive)

        Returns:
            Profile dict with metadata and R2 key
        """
        username_lower = username.lower().lstrip('@')

        if username_lower not in self.username_to_id:
            raise ValueError(f"Username '{username}' not found in metadata.\n"
                           f"Available: {len(self.username_to_id)} profiles")

        profile_id = self.username_to_id[username_lower]
        return self.profiles[profile_id]

    def fetch_image(self, username: str, save_path: str = None) -> Image.Image:
        """
        Fetch profile image from R2 by username.

        Args:
            username: Instagram username
            save_path: Optional path to save the image locally

        Returns:
            PIL Image
        """
        profile = self.get_profile_by_username(username)
        r2_key = profile.get('original_image_r2_key')

        if not r2_key:
            raise ValueError(f"No R2 key found for {username}")

        print(f"Fetching {username} from R2: {r2_key}")

        # Fetch from R2
        response = self.s3_client.get_object(
            Bucket=self.bucket,
            Key=r2_key
        )

        image_bytes = response['Body'].read()
        image = Image.open(BytesIO(image_bytes)).convert('RGB')

        print(f"✓ Fetched image: {image.size[0]}x{image.size[1]}")

        # Save locally if path provided
        if save_path:
            os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
            image.save(save_path)
            print(f"✓ Saved to: {save_path}")

        return image

    def search_usernames(self, query: str, limit: int = 10) -> list[str]:
        """Search for usernames containing the query string."""
        query_lower = query.lower()
        matches = [
            uname for uname in self.username_to_id.keys()
            if query_lower in uname
        ]
        return sorted(matches)[:limit]

    def list_usernames(self, limit: int = 20) -> list[str]:
        """List available usernames."""
        return sorted(self.username_to_id.keys())[:limit]

# Initialize the R2 fetcher with Google Drive metadata
from google.colab import drive
if not os.path.exists('/content/drive'):
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print(f"Warning: Could not mount Drive: {e}")

METADATA_PATH = "/content/drive/MyDrive/Loras/wesleygram/profiles_metadata.json"

if os.path.exists(METADATA_PATH):
    print(f"Using metadata from Drive: {METADATA_PATH}")
    r2_fetcher = R2ProfileFetcher(r2_creds, metadata_path=METADATA_PATH)
else:
    print(f"⚠ Metadata not found on Drive at {METADATA_PATH}. Falling back to R2 default.")
    r2_fetcher = R2ProfileFetcher(r2_creds)

# Show some available usernames
print(f"\nSample usernames available:")
for uname in r2_fetcher.list_usernames(10):
    print(f"  - {uname}")

In [ ]:
import torch
import gc
import os
import json
import requests
import cv2
import boto3
from io import BytesIO
from pathlib import Path
from PIL import Image, ImageFilter, ImageDraw
import numpy as np

from diffusers import (
    StableDiffusionXLInpaintPipeline,
    DPMSolverMultistepScheduler,
)
from diffusers.utils import load_image, make_image_grid

# Segmentation imports
from segment_anything import sam_model_registry, SamPredictor
from transformers import pipeline, AutoProcessor, AutoModelForZeroShotObjectDetection

# Face detection fallback
try:
    from facenet_pytorch import MTCNN
    MTCNN_AVAILABLE = True
except ImportError:
    MTCNN_AVAILABLE = False
    print("⚠ facenet-pytorch not available. Install with: pip install facenet-pytorch")

# Landmark-based face parsing
try:
    import face_recognition
    FACE_RECOGNITION_AVAILABLE = True
except ImportError:
    FACE_RECOGNITION_AVAILABLE = False
    print("⚠ face_recognition not available. Install with: pip install face_recognition")

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


def preprocess_for_detection(pil_img, clip_limit=2.0, tile_grid=(8, 8), gamma=1.1):
    """Apply CLAHE and gamma correction to improve face detection in poor lighting."""
    img = np.array(pil_img)
    if img.shape[2] == 4:  # RGBA
        img = img[:, :, :3]

    # Convert to BGR for OpenCV
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    # Apply CLAHE to L channel in LAB color space
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l_enhanced = clahe.apply(l)
    lab_enhanced = cv2.merge((l_enhanced, a, b))
    img_enhanced = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2BGR)

    # Apply gamma correction
    img_enhanced = img_enhanced.astype(np.float32) / 255.0
    img_enhanced = np.clip(img_enhanced ** (1.0 / gamma), 0, 1.0)
    img_enhanced = (img_enhanced * 255).astype(np.uint8)

    # Convert back to RGB
    img_rgb = cv2.cvtColor(img_enhanced, cv2.COLOR_BGR2RGB)
    return Image.fromarray(img_rgb)


def nms_boxes(boxes, scores, iou_threshold=0.5):
    """Non-maximum suppression to remove duplicate boxes."""
    if not boxes:
        return [], []

    boxes = np.array(boxes)
    scores = np.array(scores)

    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]

    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)

        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        w = np.maximum(0.0, xx2 - xx1)
        h = np.maximum(0.0, yy2 - yy1)
        inter = w * h

        iou = inter / (areas[i] + areas[order[1:]] - inter)
        inds = np.where(iou <= iou_threshold)[0]
        order = order[inds + 1]

    return boxes[keep].tolist(), scores[keep].tolist()


class PersonSegmenter:
    """
    Person/Face segmenter using Grounding DINO + SAM + MTCNN fallback.
    Supports 'face' (default) and 'body' targets.

    Features:
    - CLAHE + gamma preprocessing for better detection in poor lighting
    - MTCNN face detector fallback for small/dark faces
    - Multi-scale detection for different face sizes
    - Glasses detection and union to prevent holes in mask
    - face_recognition landmarks to keep mask to face + neck
    """

    def __init__(
        self,
        sam_checkpoint: str,
        sam_model_type: str = "vit_h",
        detection_model: str = "IDEA-Research/grounding-dino-tiny",
        device: str = "cuda",
        verbose: bool = True
    ):
        self.device = device
        self.verbose = verbose
        self._sam = None
        self._sam_predictor = None
        self._detector_processor = None
        self._detector_model = None
        self._mtcnn = None
        self.sam_checkpoint = sam_checkpoint
        self.sam_model_type = sam_model_type
        self.detection_model_name = detection_model

    def _load_sam(self):
        """Lazy load SAM model."""
        if self._sam is None:
            if self.verbose: print("Loading SAM model...")
            self._sam = sam_model_registry[self.sam_model_type](checkpoint=self.sam_checkpoint)
            self._sam.to(self.device)
            self._sam_predictor = SamPredictor(self._sam)
            if self.verbose: print("✓ SAM loaded")

    def _load_detector(self):
        """Lazy load Grounding DINO."""
        if self._detector_model is None:
            if self.verbose: print("Loading Grounding DINO...")
            self._detector_processor = AutoProcessor.from_pretrained(self.detection_model_name)
            self._detector_model = AutoModelForZeroShotObjectDetection.from_pretrained(
                self.detection_model_name
            ).to(self.device)
            if self.verbose: print("✓ Grounding DINO loaded")

    def _load_mtcnn(self):
        """Lazy load MTCNN face detector."""
        if self._mtcnn is None and MTCNN_AVAILABLE:
            if self.verbose: print("Loading MTCNN face detector...")
            self._mtcnn = MTCNN(
                keep_all=True,
                device=self.device,
                min_face_size=20,
                thresholds=[0.6, 0.7, 0.7]
            )
            if self.verbose: print("✓ MTCNN loaded")

    def detect_faces_mtcnn(
        self,
        image: Image.Image,
        min_prob: float = 0.85,
        min_face_size: int = 15
    ) -> tuple[list[list[float]], list[float]]:
        """Detect faces using MTCNN (good for small/occluded faces)."""
        if not MTCNN_AVAILABLE:
            return [], []

        self._load_mtcnn()

        boxes, probs = self._mtcnn.detect(image)

        if boxes is None or len(boxes) == 0:
            return [], []

        filtered_boxes = []
        filtered_scores = []

        for box, prob in zip(boxes, probs):
            if prob is None or prob < min_prob:
                continue

            x1, y1, x2, y2 = box.tolist()
            width = x2 - x1
            height = y2 - y1

            if width < min_face_size or height < min_face_size:
                continue

            filtered_boxes.append([x1, y1, x2, y2])
            filtered_scores.append(float(prob))

        if self.verbose and filtered_boxes:
            print(f"MTCNN detected {len(filtered_boxes)} faces")

        return filtered_boxes, filtered_scores

    def detect_people(
        self,
        image: Image.Image,
        threshold: float = 0.3,
        target: str = "face"
    ) -> list[list[float]]:
        """Detect objects in image using Grounding DINO."""
        self._load_detector()

        target_l = target.lower().strip()

        if target_l in ["face", "head"]:
            text_prompt = "head. face. chin."
        elif target_l in ["glasses", "eyeglasses", "sunglasses"]:
            text_prompt = "glasses. sunglasses."
        else:
            text_prompt = "person. human."

        if self.verbose: print(f"Detecting with prompt: '{text_prompt}'")

        inputs = self._detector_processor(
            images=image,
            text=text_prompt,
            return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():
            outputs = self._detector_model(**inputs)

        results = self._detector_processor.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold,
            threshold,
            [image.size[::-1]]
        )[0]

        boxes = results["boxes"].cpu().numpy().tolist()
        scores = results["scores"].cpu().numpy().tolist()

        if self.verbose:
            print(f"Grounding DINO detected {len(boxes)} object(s)")
            for i, (box, score) in enumerate(zip(boxes, scores)):
                print(f"  Box {i}: score={score:.3f}, coords={[round(c) for c in box]}")

        return boxes

    def detect_multiscale(
        self,
        image: Image.Image,
        threshold: float = 0.3,
        target: str = "face",
        scales: list[float] = [0.5, 1.0, 1.5],
        use_mtcnn: bool = True,
        use_preprocessing: bool = True
    ) -> tuple[list[list[float]], list[float]]:
        """Detect faces at multiple scales and combine results."""
        all_boxes = []
        all_scores = []

        original_w, original_h = image.size

        # Preprocess image for better detection in poor lighting
        if use_preprocessing:
            preprocessed = preprocess_for_detection(image)
        else:
            preprocessed = image

        # Multi-scale Grounding DINO detection
        for scale in scales:
            if scale != 1.0:
                new_w = int(original_w * scale)
                new_h = int(original_h * scale)
                scaled_img = preprocessed.resize((new_w, new_h), Image.LANCZOS)
            else:
                scaled_img = preprocessed

            boxes = self.detect_people(scaled_img, threshold=threshold, target=target)

            # Scale boxes back to original size
            for box in boxes:
                x1, y1, x2, y2 = box
                if scale != 1.0:
                    x1, y1, x2, y2 = x1/scale, y1/scale, x2/scale, y2/scale
                all_boxes.append([x1, y1, x2, y2])
                all_scores.append(0.9)  # Default score for Grounding DINO

        # MTCNN fallback (especially good for small faces)
        if use_mtcnn and target in ["face", "head"]:
            mtcnn_boxes, mtcnn_scores = self.detect_faces_mtcnn(preprocessed)
            all_boxes.extend(mtcnn_boxes)
            all_scores.extend(mtcnn_scores)

        if not all_boxes:
            return [], []

        # Apply NMS to remove duplicates
        boxes_nms, scores_nms = nms_boxes(all_boxes, all_scores, iou_threshold=0.5)

        if self.verbose:
            print(f"Total after NMS: {len(boxes_nms)} boxes")

        return boxes_nms, scores_nms

    def segment_from_boxes(
        self,
        image: Image.Image,
        boxes: list[list[float]],
        expansion_pixels: int = 10,
        merge_boxes: bool = True
    ) -> np.ndarray:
        """Generate segmentation mask using SAM."""
        self._load_sam()

        image_np = np.array(image)
        self._sam_predictor.set_image(image_np)

        combined_mask = np.zeros((image_np.shape[0], image_np.shape[1]), dtype=np.uint8)

        if merge_boxes and len(boxes) > 1:
            all_x1 = min(b[0] for b in boxes)
            all_y1 = min(b[1] for b in boxes)
            all_x2 = max(b[2] for b in boxes)
            all_y2 = max(b[3] for b in boxes)

            merged_width = all_x2 - all_x1
            merged_height = all_y2 - all_y1
            aspect_ratio = merged_height / merged_width if merged_width > 0 else 0

            if 0.8 <= aspect_ratio <= 2.0:
                merged_box = [all_x1, all_y1, all_x2, all_y2]
                boxes = [merged_box]
                if self.verbose: print(f"Merged boxes (aspect ratio {aspect_ratio:.2f}): {[round(c) for c in merged_box]}")
            else:
                if self.verbose: print(f"Skipping merge, aspect ratio {aspect_ratio:.2f} seems wrong")

        for box in boxes:
            x1, y1, x2, y2 = box
            x1 = max(0, x1 - expansion_pixels)
            y1 = max(0, y1 - expansion_pixels)
            x2 = min(image_np.shape[1], x2 + expansion_pixels)
            y2 = min(image_np.shape[0], y2 + expansion_pixels)

            box_np = np.array([[x1, y1, x2, y2]])

            center_x = (x1 + x2) / 2
            center_y = (y1 + y2) / 2
            point_coords = np.array([[center_x, center_y]])
            point_labels = np.array([1])

            masks, scores, _ = self._sam_predictor.predict(
                point_coords=point_coords,
                point_labels=point_labels,
                box=box_np,
                multimask_output=True
            )

            best_mask_idx = int(np.argmax(scores))
            mask = masks[best_mask_idx]
            combined_mask = np.maximum(combined_mask, (mask * 255).astype(np.uint8))

        return combined_mask

    def _mask_face_and_neck_landmarks(
        self,
        image: Image.Image,
        neck_ratio: float = 0.35,
        feather_radius: int = 5
    ) -> Image.Image | None:
        """Use face_recognition landmarks to keep the mask to face + neck."""
        if not FACE_RECOGNITION_AVAILABLE:
            return None

        img_np = np.array(image.convert("RGB"))
        face_locations = face_recognition.face_locations(img_np)

        if not face_locations:
            if self.verbose:
                print("⚠ face_recognition could not find a face.")
            return None

        landmarks_list = face_recognition.face_landmarks(img_np, face_locations)
        mask = Image.new("L", image.size, 0)
        drawer = ImageDraw.Draw(mask)

        for landmarks in landmarks_list:
            chin = landmarks.get("chin")
            if not chin:
                continue

            face_height = max(y for _, y in chin) - min(y for _, y in chin)
            neck_extension = max(10, int(face_height * neck_ratio))

            left = chin[0]
            right = chin[-1]
            center = chin[len(chin) // 2]

            neck_points = [
                (left[0], left[1] + neck_extension),
                (center[0], center[1] + neck_extension),
                (right[0], right[1] + neck_extension),
            ]

            polygon = chin + neck_points[::-1]
            drawer.polygon(polygon, fill=255)

        if mask.getbbox() is None:
            if self.verbose:
                print("⚠ face_recognition landmarks empty; falling back to SAM.")
            return None

        if feather_radius > 0:
            mask = mask.filter(ImageFilter.GaussianBlur(radius=max(1, feather_radius)))

        mask_np = np.array(mask)
        mask_np = np.where(mask_np > 32, 255, 0).astype(np.uint8)
        mask_pil = Image.fromarray(mask_np).filter(ImageFilter.MaxFilter(3))
        return mask_pil

    def _bbox_face_neck_mask(self, image: Image.Image, box: list[float], neck_ratio: float = 0.35) -> np.ndarray:
        """Clamp mask to face bbox plus a neck extension to avoid including torso."""
        x1, y1, x2, y2 = box
        w = max(1, x2 - x1)
        h = max(1, y2 - y1)
        neck_extension = max(10, int(h * neck_ratio))

        x1i = int(max(0, np.floor(x1)))
        y1i = int(max(0, np.floor(y1)))
        x2i = int(min(image.size[0], np.ceil(x2)))
        y2i = int(min(image.size[1], np.ceil(y2 + neck_extension)))

        mask = np.zeros((image.size[1], image.size[0]), dtype=np.uint8)
        mask[y1i:y2i, x1i:x2i] = 255
        return mask

    def generate_mask(
        self,
        image: Image.Image,
        detection_threshold: float = 0.3,
        expansion_pixels: int = 10,
        feather_radius: int = 5,
        target: str = "face",
        use_multiscale: bool = True,
        use_mtcnn: bool = True,
        use_preprocessing: bool = True
    ) -> Image.Image:
        """Full pipeline: detect → segment → feather edges."""

        target_l = target.lower().strip()

        if target_l in ["face", "head"]:
            landmark_mask = self._mask_face_and_neck_landmarks(
                image,
                neck_ratio=0.35,
                feather_radius=feather_radius
            )
            if landmark_mask is not None:
                return landmark_mask

            # Multi-scale + MTCNN detection
            if use_multiscale:
                face_boxes, face_scores = self.detect_multiscale(
                    image,
                    threshold=detection_threshold,
                    target="face",
                    scales=[0.5, 1.0, 1.5],
                    use_mtcnn=use_mtcnn,
                    use_preprocessing=use_preprocessing
                )
            else:
                face_boxes = self.detect_people(image, threshold=detection_threshold, target="face")
                face_scores = [0.9 for _ in face_boxes]

            # Fallback to lower threshold
            if not face_boxes and detection_threshold > 0.15:
                if self.verbose: print("⚠ No face/head detections, retrying with threshold=0.15...")
                face_boxes = self.detect_people(image, threshold=0.15, target="face")
                face_scores = [0.9 for _ in face_boxes]

            # Fallback to person detection
            if not face_boxes:
                if self.verbose: print("⚠ Still no face/head detections, trying 'person' target...")
                face_boxes = self.detect_people(image, threshold=0.2, target="person")
                face_scores = [0.9 for _ in face_boxes]

            if not face_boxes:
                if self.verbose: print("⚠ No objects detected!")
                return Image.new("L", image.size, 0)

            # Keep the highest-confidence face box only to avoid pulling torso/background
            if face_scores:
                best_idx = int(np.argmax(face_scores))
            else:
                best_idx = 0
            primary_box = face_boxes[best_idx]
            face_expansion = max(expansion_pixels, 12)
            face_mask_np = self.segment_from_boxes(
                image,
                [primary_box],
                expansion_pixels=face_expansion,
                merge_boxes=False
            )

            # Glasses detection (union)
            glasses_threshold = max(0.2, detection_threshold - 0.05)
            glasses_boxes = self.detect_people(image, threshold=glasses_threshold, target="glasses")

            if glasses_boxes:
                glasses_expansion = max(6, expansion_pixels // 2)
                glasses_mask_np = self.segment_from_boxes(
                    image,
                    glasses_boxes,
                    expansion_pixels=glasses_expansion,
                    merge_boxes=False
                )
                mask_np = np.maximum(face_mask_np, glasses_mask_np)
            else:
                mask_np = face_mask_np

            # Clamp to face bbox + neck extension to avoid torso
            clamp_np = self._bbox_face_neck_mask(image, primary_box, neck_ratio=0.4)
            mask_np = np.minimum(mask_np, clamp_np)

        else:
            boxes = self.detect_people(image, threshold=detection_threshold, target=target)

            if not boxes and detection_threshold > 0.15:
                if self.verbose: print("⚠ No objects detected, retrying with threshold=0.15...")
                boxes = self.detect_people(image, threshold=0.15, target=target)

            if not boxes:
                if self.verbose: print("⚠ Still no detections, trying 'person' target...")
                boxes = self.detect_people(image, threshold=0.2, target="person")

            if not boxes:
                if self.verbose: print("⚠ No objects detected!")
                return Image.new("L", image.size, 0)

            mask_np = self.segment_from_boxes(image, boxes, expansion_pixels, merge_boxes=True)

        # Feather edges
        mask_pil = Image.fromarray(mask_np)
        if feather_radius > 0:
            mask_pil = mask_pil.filter(ImageFilter.GaussianBlur(radius=feather_radius))
            mask_np2 = np.array(mask_pil)
            mask_np2 = np.where(mask_np2 > 64, 255, 0).astype(np.uint8)
            mask_pil = Image.fromarray(mask_np2)
            mask_pil = mask_pil.filter(ImageFilter.GaussianBlur(radius=max(1, feather_radius // 2)))

        return mask_pil

    def generate_person_mask(self, image, detection_threshold=0.3, expansion_pixels=10, feather_radius=5):
        """Backward compatibility alias."""
        return self.generate_mask(image, detection_threshold, expansion_pixels, feather_radius, target="face")

    def cleanup(self):
        """Free GPU memory."""
        if self._sam_predictor: del self._sam_predictor
        if self._sam: del self._sam
        if self._detector_model: del self._detector_model
        if self._detector_processor: del self._detector_processor
        if self._mtcnn: del self._mtcnn
        self._sam = None
        self._sam_predictor = None
        self._detector_model = None
        self._detector_processor = None
        self._mtcnn = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            if self.verbose: print(f"✓ Cleaned up. VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Initialize the test segmenter
print("Initializing segmenter (models load on first use)...")
test_segmenter = PersonSegmenter(
    sam_checkpoint=SAM_CHECKPOINT,
    sam_model_type=SAM_MODEL_TYPE,
    detection_model=DETECTION_MODEL,
    device=DEVICE
)
print("✓ Segmenter ready")

In [ ]:
# =============================================================================
# Upload LoRA file (required for inpainting)
# =============================================================================

# Mount Google Drive to access the LoRA file
from google.colab import drive
drive.mount('/content/drive')

# Set the LORA_PATH to the user's specified Google Drive path
LORA_PATH = "/content/drive/MyDrive/Loras/wesleygram/output/wesleygram-25.safetensors"

print(f"✓ LoRA path set to: {LORA_PATH}")

# Verify that the LoRA file exists
import os
if not os.path.exists(LORA_PATH):
    raise FileNotFoundError(f"⚠ Warning: LoRA file not found at {LORA_PATH}. Please check the path and try again.")
else:
    print("✓ LoRA file found at specified path.")


---
# 🧪 TESTING SECTION: Mask Generation
---

This section allows you to test the person segmentation pipeline independently:
1. Enter an Instagram username
2. Fetch the profile photo from R2
3. Generate a person mask using ML
4. Display both the original image and the generated mask

In [ ]:
# =============================================================================
# TEST: Fetch profile image by username
# =============================================================================

# Enter the Instagram username to test
TEST_USERNAME = "micha.el.j"  # @param {type:"string"}

# Fetch the image from R2
print(f"Fetching profile photo for: @{TEST_USERNAME}")
test_image_path = f"/content/test_{TEST_USERNAME}.jpg"

try:
    test_image = r2_fetcher.fetch_image(TEST_USERNAME, save_path=test_image_path)

    # Get profile metadata
    profile_data = r2_fetcher.get_profile_by_username(TEST_USERNAME)
    print(f"\nProfile Info:")
    print(f"  Full Name: {profile_data.get('full_name', 'N/A')}")
    print(f"  Bio: {profile_data.get('biography', 'N/A')[:50]}...")
    print(f"  Followers: {profile_data.get('follower_count', 'N/A')}")
    print(f"  Private: {profile_data.get('is_private', 'N/A')}")

    # Display the image
    print(f"\nOriginal Image ({test_image.size[0]}x{test_image.size[1]}):")
    display(test_image)

except Exception as e:
    print(f"❌ Error: {e}")
    test_image = None

In [ ]:
# =============================================================================
# TEST: Generate and Display Mask
# =============================================================================

if test_image is not None:
    print(f"Generating person mask for @{TEST_USERNAME}...")
    print("-" * 50)

    # Generate mask
    test_mask = test_segmenter.generate_mask(
        test_image,
        detection_threshold=PERSON_DETECTION_THRESHOLD,
        expansion_pixels=MASK_EXPANSION_PIXELS,
        feather_radius=MASK_FEATHER_RADIUS
    )

    # Save mask
    test_mask_path = f"/content/test_{TEST_USERNAME}_mask.png"
    test_mask.save(test_mask_path)
    print(f"\n✓ Mask saved to: {test_mask_path}")

    # Create visualization: Original | Mask | Overlay
    print("\nResults (Original | Mask | Overlay):")

    # Create overlay visualization
    overlay = test_image.copy().convert("RGBA")
    mask_rgba = Image.new("RGBA", test_mask.size, (255, 0, 0, 0))
    mask_np = np.array(test_mask)
    mask_alpha = (mask_np > 128).astype(np.uint8) * 128  # Semi-transparent red
    mask_rgba_np = np.zeros((*mask_np.shape, 4), dtype=np.uint8)
    mask_rgba_np[:, :, 0] = 255  # Red channel
    mask_rgba_np[:, :, 3] = mask_alpha  # Alpha channel
    mask_rgba = Image.fromarray(mask_rgba_np, mode="RGBA")
    overlay = Image.alpha_composite(overlay, mask_rgba).convert("RGB")

    # Display grid
    display(make_image_grid([test_image, test_mask.convert("RGB"), overlay], rows=1, cols=3))

    print(f"\n✓ Mask coverage: {np.mean(np.array(test_mask) > 128) * 100:.1f}% of image")
else:
    print("❌ No test image loaded. Run the previous cell first.")

In [ ]:
# =============================================================================
# TEST: Interactive Testing - Try Multiple Usernames
# =============================================================================

def test_mask_for_username(username: str, display_results: bool = True):
    """
    Convenience function to test mask generation for any username.

    Args:
        username: Instagram username to test
        display_results: Whether to display the images

    Returns:
        Tuple of (image, mask) or None if failed
    """
    try:
        # Fetch image
        print(f"{'='*50}")
        print(f"Testing: @{username}")
        print(f"{'='*50}")

        image = r2_fetcher.fetch_image(username)

        # Generate mask
        mask = test_segmenter.generate_mask(
            image,
            detection_threshold=PERSON_DETECTION_THRESHOLD,
            expansion_pixels=MASK_EXPANSION_PIXELS,
            feather_radius=MASK_FEATHER_RADIUS
        )

        if display_results:
            # Create overlay
            overlay = image.copy().convert("RGBA")
            mask_np = np.array(mask)
            mask_rgba_np = np.zeros((*mask_np.shape, 4), dtype=np.uint8)
            mask_rgba_np[:, :, 0] = 255
            mask_rgba_np[:, :, 3] = (mask_np > 128).astype(np.uint8) * 128
            mask_rgba = Image.fromarray(mask_rgba_np, mode="RGBA")
            overlay = Image.alpha_composite(overlay, mask_rgba).convert("RGB")

            display(make_image_grid([image, mask.convert("RGB"), overlay], rows=1, cols=3))
            print(f"Mask coverage: {np.mean(mask_np > 128) * 100:.1f}%")

        return image, mask

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# Example: Test a few different usernames
# Uncomment to run:
# test_mask_for_username("15harshit")
# test_mask_for_username("some_other_user")

In [ ]:
# =============================================================================
# TEST: Search for usernames
# =============================================================================

# Search for usernames containing a string
search_query = "kai"  # @param {type:"string"}
print(f"Searching for usernames containing '{search_query}':")
matches = r2_fetcher.search_usernames(search_query, limit=15)
for match in matches:
    print(f"  - {match}")

In [ ]:
# =============================================================================
# TEST: Cleanup segmenter to free memory (optional)
# Run this before loading SDXL models if needed
# =============================================================================

# Uncomment to cleanup test segmenter:
# test_segmenter.cleanup()
# del test_segmenter
# print("✓ Test segmenter cleaned up")

---
# 🎨 MAIN PIPELINE: SDXL Inpainting
---

The sections below implement the full inpainting pipeline with SDXL base + refiner models.

## 4. Person Segmentation (Full Pipeline)

In [ ]:
def preprocess_image_and_mask(
    image_path: str,
    mask_path: str,
    target_size: int = 1024,
    feather_radius: int = 5
) -> tuple[Image.Image, Image.Image]:
    """
    Preprocess input image and mask for SDXL inpainting.

    Args:
        image_path: Path to the input image
        mask_path: Path to the mask image
        target_size: Target resolution (1024 for SDXL)
        feather_radius: Radius for edge feathering to avoid seams

    Returns:
        Tuple of (processed_image, processed_mask)

    Mask behavior:
        - White (255) = regions to regenerate (person to replace)
        - Black (0) = regions to preserve (background)
    """
    # Load images
    image = Image.open(image_path).convert("RGB")
    mask = Image.open(mask_path).convert("L")  # Single channel grayscale

    original_size = image.size
    print(f"Original image size: {original_size}")
    print(f"Original mask size: {mask.size}")

    # Resize image to target resolution (maintain aspect ratio, center crop)
    # For best results with SDXL, use 1024x1024
    aspect = image.width / image.height

    if aspect > 1:  # Wider than tall
        new_width = int(target_size * aspect)
        new_height = target_size
    else:  # Taller than wide
        new_width = target_size
        new_height = int(target_size / aspect)

    # Resize with high-quality resampling
    image = image.resize((new_width, new_height), Image.Resampling.LANCZOS)

    # Center crop to target_size x target_size
    left = (new_width - target_size) // 2
    top = (new_height - target_size) // 2
    image = image.crop((left, top, left + target_size, top + target_size))

    # Resize mask to match image using NEAREST to preserve hard edges
    # Then apply the same crop
    mask = mask.resize((new_width, new_height), Image.Resampling.NEAREST)
    mask = mask.crop((left, top, left + target_size, top + target_size))

    # Apply slight feathering to mask edges to avoid seams
    if feather_radius > 0:
        from PIL import ImageFilter
        # Blur the mask slightly for soft edges
        mask_array = np.array(mask, dtype=np.float32)
        mask_pil = Image.fromarray(mask_array.astype(np.uint8))
        mask_blurred = mask_pil.filter(ImageFilter.GaussianBlur(radius=feather_radius))
        # Blend: keep hard mask in center, soft edges on boundary
        mask_array_blurred = np.array(mask_blurred, dtype=np.float32)
        # Threshold to maintain mostly binary mask with soft edges
        mask_array_blurred = np.clip(mask_array_blurred, 0, 255)
        mask = Image.fromarray(mask_array_blurred.astype(np.uint8))

    print(f"Processed image size: {image.size}")
    print(f"Processed mask size: {mask.size}")

    return image, mask

In [ ]:
# Initialize the full PersonSegmenter for the main pipeline
# (This is separate from the test segmenter)

segmenter = PersonSegmenter(
    sam_checkpoint=SAM_CHECKPOINT,
    sam_model_type=SAM_MODEL_TYPE,
    detection_model=DETECTION_MODEL,
    device=DEVICE
)

## 5. Fetch Image from R2 and Generate Mask

Enter a username to fetch their profile photo from R2 and generate a person mask.

In [ ]:
def preprocess_image_and_mask_from_pil(
    image: Image.Image,
    mask: Image.Image = None,
    segmenter = None,
    target_size: int = 1024,
    feather_radius: int = 5,
    detection_threshold: float = 0.3,
    expansion_pixels: int = 10
) -> tuple[Image.Image, Image.Image]:
    """
    Preprocess PIL image and mask for SDXL inpainting.

    If no mask is provided, automatically generates one using ML-based
    person segmentation (Grounding DINO + SAM).

    Args:
        image: PIL Image (already loaded)
        mask: PIL Image mask (optional - auto-generate if None)
        segmenter: PersonSegmenter instance for auto-mask generation
        target_size: Target resolution (1024 for SDXL)
        feather_radius: Radius for edge feathering to avoid seams
        detection_threshold: Confidence for person detection
        expansion_pixels: Expand mask beyond detected person

    Returns:
        Tuple of (processed_image, processed_mask)
    """
    original_size = image.size
    print(f"Original image size: {original_size}")

    # Generate or use provided mask
    if mask is None and segmenter is not None:
        print("Auto-generating person mask using ML segmentation...")
        mask = segmenter.generate_person_mask(
            image,
            detection_threshold=detection_threshold,
            expansion_pixels=expansion_pixels,
            feather_radius=feather_radius
        )
        print("Mask generated successfully!")
    elif mask is None:
        raise ValueError("Either mask or segmenter must be provided!")

    print(f"Mask size: {mask.size}")

    # Resize image to target resolution (maintain aspect ratio, center crop)
    aspect = image.width / image.height

    if aspect > 1:
        new_width = int(target_size * aspect)
        new_height = target_size
    else:
        new_width = target_size
        new_height = int(target_size / aspect)

    # Resize with high-quality resampling
    image = image.resize((new_width, new_height), Image.Resampling.LANCZOS)

    # Center crop
    left = (new_width - target_size) // 2
    top = (new_height - target_size) // 2
    image = image.crop((left, top, left + target_size, top + target_size))

    # Resize and crop mask to match
    mask = mask.resize((new_width, new_height), Image.Resampling.NEAREST)
    mask = mask.crop((left, top, left + target_size, top + target_size))

    print(f"Processed image size: {image.size}")
    print(f"Processed mask size: {mask.size}")

    return image, mask

In [ ]:
# =============================================================================
# Fetch image from R2 and generate mask
# =============================================================================

# Enter the username for inpainting
INPAINT_USERNAME = "5starr.valleriee"  # @param {type:"string"}

print(f"Fetching profile photo for inpainting: @{INPAINT_USERNAME}")
print("=" * 50)

try:
    # Fetch from R2
    raw_image = r2_fetcher.fetch_image(INPAINT_USERNAME)

    # Preprocess and generate mask
    input_image, mask_image = preprocess_image_and_mask_from_pil(
        image=raw_image,
        mask=None,  # Auto-generate
        segmenter=segmenter,
        target_size=RESOLUTION,
        feather_radius=MASK_FEATHER_RADIUS,
        detection_threshold=PERSON_DETECTION_THRESHOLD,
        expansion_pixels=MASK_EXPANSION_PIXELS
    )

    # Display preprocessed images
    print("\nInput Image | Generated Mask:")
    display(make_image_grid([input_image, mask_image], rows=1, cols=2))

    # Save locally for reference
    input_image.save(f"/content/{INPAINT_USERNAME}_input.png")
    mask_image.save(f"/content/{INPAINT_USERNAME}_mask.png")
    print(f"\n✓ Saved input and mask to /content/")

except Exception as e:
    print(f"❌ Error: {e}")
    input_image = None
    mask_image = None

In [ ]:
# Free up segmentation model memory before loading SDXL
print("Cleaning up segmentation models to free VRAM...")
segmenter.cleanup()
del segmenter
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 5. Load Models

In [ ]:
def load_inpaint_pipelines(
    base_model: str,
    refiner_model: str,
    lora_path: str,
    lora_scale: float,
    device: str,
    dtype: torch.dtype
) -> tuple:
    """
    Load SDXL base and refiner inpainting pipelines with optimizations.

    Uses shared text_encoder_2 and vae between base and refiner to save memory.
    """
    print("Loading base inpainting model...")

    # Load base model
    base = StableDiffusionXLInpaintPipeline.from_pretrained(
        base_model,
        torch_dtype=dtype,
        variant="fp16",
        use_safetensors=True,
        add_watermarker=False,  # Disable watermarking
    )

    # Configure DPM++ scheduler for efficient sampling
    base.scheduler = DPMSolverMultistepScheduler.from_config(
        base.scheduler.config,
        algorithm_type="dpmsolver++",
        use_karras_sigmas=True,
    )

    print("Loading refiner model (sharing VAE and text_encoder_2)...")

    # Load refiner with shared components to save memory
    refiner = StableDiffusionXLInpaintPipeline.from_pretrained(
        refiner_model,
        text_encoder_2=base.text_encoder_2,
        vae=base.vae,
        torch_dtype=dtype,
        variant="fp16",
        use_safetensors=True,
        add_watermarker=False,
    )

    # Same scheduler for refiner
    refiner.scheduler = DPMSolverMultistepScheduler.from_config(
        refiner.scheduler.config,
        algorithm_type="dpmsolver++",
        use_karras_sigmas=True,
    )

    # Load LoRA weights into base model
    print(f"Loading LoRA from: {lora_path}")
    lora_file = Path(lora_path)

    if not lora_file.exists():
        raise FileNotFoundError(f"LoRA file not found: {lora_path}")

    # Load LoRA - works with both local files and HF repos
    base.load_lora_weights(
        str(lora_file.parent),
        weight_name=lora_file.name
    )

    # NOTE: We do NOT load LoRA into the refiner because the architectures differ
    # (Base has 640 channels at block 1, Refiner has 768).
    # The LoRA is trained on Base and is incompatible with Refiner.
    # refiner.load_lora_weights(...) <--- Removed

    print(f"LoRA loaded into Base model with scale: {lora_scale}")

    # Apply memory optimizations
    print("Applying optimizations...")

    if device == "cuda":
        # Move to GPU
        base.to(device)
        refiner.to(device)

        # Enable memory efficient attention
        try:
            base.enable_xformers_memory_efficient_attention()
            refiner.enable_xformers_memory_efficient_attention()
            print("✓ xFormers memory efficient attention enabled")
        except Exception as e:
            print(f"xFormers not available: {e}")

        # Enable CUDA optimizations
        torch.backends.cudnn.benchmark = True

        # Try torch.compile for additional speedup (PyTorch 2.x)
        try:
            base.unet = torch.compile(base.unet, mode="reduce-overhead", fullgraph=False)
            refiner.unet = torch.compile(refiner.unet, mode="reduce-overhead", fullgraph=False)
            print("✓ torch.compile enabled on UNet")
        except Exception as e:
            print(f"torch.compile not available: {e}")
    else:
        # CPU optimizations
        base.enable_attention_slicing()
        refiner.enable_attention_slicing()
        base.vae.enable_slicing()
        print("✓ Attention slicing enabled for CPU")

    print("Models loaded successfully!")

    return base, refiner

In [ ]:
# Clear any existing models from memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load the pipelines
base_pipe, refiner_pipe = load_inpaint_pipelines(
    base_model=BASE_MODEL,
    refiner_model=REFINER_MODEL,
    lora_path=LORA_PATH,
    lora_scale=LORA_SCALE,
    device=DEVICE,
    dtype=DTYPE
)

print(f"\nVRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB" if torch.cuda.is_available() else "")

## 6. Inpainting Function

In [ ]:
def inpaint_with_lora(
    base_pipe,
    refiner_pipe,
    image: Image.Image,
    mask: Image.Image,
    prompt: str,
    negative_prompt: str,
    num_inference_steps: int = 35,
    guidance_scale: float = 9.0,
    high_noise_frac: float = 0.95,  # CHANGED: Default to 0.95 for identity preservation
    strength: float = 0.65,
    lora_scale: float = 0.85,
    seed: int = -1,
    device: str = "cuda",
    verbose: bool = True
) -> Image.Image:
    """
    Perform SDXL inpainting using base + refiner (optional).
    Supports low-strength face replacement mode.
    """
    # Set up generator for reproducibility
    if seed == -1:
        seed = torch.randint(0, 2**32, (1,)).item()

    generator = torch.Generator(device=device).manual_seed(seed)
    if verbose:
        print(f"Using seed: {seed}")
        print(f"Inpainting Strength: {strength}")

    # Ensure mask is in correct format
    if mask.mode != "L":
        mask = mask.convert("L")

    with torch.inference_mode():
        # Strategy: Use Base model for majority of denoising (with LoRA)
        # Then optionally hand off to Refiner for final details (texture)

        use_refiner = refiner_pipe is not None

        if verbose:
            print(f"Executing Inpainting (strength={strength})...")
            print(f"Refiner enabled: {use_refiner}")

        if use_refiner:
            # Stage 1: Base Model -> Latents
            # Denoise until 'high_noise_frac' point (e.g. 0.95)
            if verbose: print(f"Stage 1: Base Model (Steps 0-{int(high_noise_frac*100)}%)...")
            latents = base_pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                image=image,
                mask_image=mask,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                strength=strength,
                denoising_end=high_noise_frac,
                output_type="latent",
                generator=generator,
                cross_attention_kwargs={"scale": lora_scale},
            ).images

            # Stage 2: Refiner Model -> PIL
            # Finish denoising from 'high_noise_frac' to 0
            if verbose: print(f"Stage 2: Refiner Model (Steps {int(high_noise_frac*100)}-100%)...")
            result = refiner_pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                image=latents,
                mask_image=mask,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                denoising_start=high_noise_frac,
                generator=generator,
            ).images[0]

        else:
            # Base Model Only (Standard)
            result = base_pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                image=image,
                mask_image=mask,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                strength=strength,
                output_type="pil",
                generator=generator,
                cross_attention_kwargs={"scale": lora_scale},
            ).images[0]

    if verbose: print("Inpainting complete!")
    return result, seed

## 8. Run Inpainting

In [ ]:
# =============================================================================
# DYNAMIC POSE DETECTION (BLIP-VQA)
# =============================================================================

from transformers import BlipProcessor, BlipForQuestionAnswering

print("Loading BLIP-VQA model for dynamic pose analysis...")

# Load lightweight VQA model
vqa_processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
vqa_model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base").to(DEVICE)

print("✓ BLIP-VQA loaded!")

def get_dynamic_pose_description(image: Image.Image) -> str:
    """
    Analyze image using VQA to determine gaze, head, body, expression, and details.
    Returns a comma-separated description string matching specific training tags.
    """
    def clean_answer(ans):
        return ans.lower().strip().replace('.', '')

    # 1. Gaze & Facing Analysis
    # First check if facing away (back of head)
    inputs = vqa_processor(image, "Is the person facing away from the camera?", return_tensors="pt").to(DEVICE)
    out = vqa_model.generate(**inputs)
    facing_away = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))

    looking_tag = ""
    if "yes" in facing_away:
        looking_tag = "looking away"
    else:
        # Explicitly check for eye contact
        inputs = vqa_processor(image, "Are the eyes looking at the camera?", return_tensors="pt").to(DEVICE)
        out = vqa_model.generate(**inputs)
        eye_contact_ans = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))

        if "yes" in eye_contact_ans:
            looking_tag = "looking at camera"
        else:
            # If not looking at camera, check direction
            inputs = vqa_processor(image, "Where are the eyes looking? left, right, up, or down?", return_tensors="pt").to(DEVICE)
            out = vqa_model.generate(**inputs)
            gaze_raw = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))

            looking_tag = "looking at camera" # Default fallback
            if "left" in gaze_raw:
                looking_tag = "looking left"
            elif "right" in gaze_raw:
                looking_tag = "looking right"
            elif "up" in gaze_raw:
                looking_tag = "looking up"
            elif "down" in gaze_raw:
                looking_tag = "looking down"

    # 2. Head Analysis
    inputs = vqa_processor(image, "Is the head turned left, right, up, or down?", return_tensors="pt").to(DEVICE)
    out = vqa_model.generate(**inputs)
    head_raw = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))

    head_tag = ""
    if "left" in head_raw:
        inputs = vqa_processor(image, "Is the head tilted or turned?", return_tensors="pt").to(DEVICE)
        out = vqa_model.generate(**inputs)
        style = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))
        head_tag = "head tilted left" if "tilt" in style else "head turned left"
    elif "right" in head_raw:
        inputs = vqa_processor(image, "Is the head tilted or turned?", return_tensors="pt").to(DEVICE)
        out = vqa_model.generate(**inputs)
        style = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))
        head_tag = "head tilted right" if "tilt" in style else "head turned right"
    elif "up" in head_raw:
        head_tag = "head pointed up"
    elif "down" in head_raw:
        head_tag = "head pointed down"

    # 3. Body Analysis
    # Check for torso visibility first
    inputs = vqa_processor(image, "Is the torso visible?", return_tensors="pt").to(DEVICE)
    out = vqa_model.generate(**inputs)
    torso_ans = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))

    body_tag = ""
    if "yes" in torso_ans:
        inputs = vqa_processor(image, "Is the body facing the camera, left, or right?", return_tensors="pt").to(DEVICE)
        out = vqa_model.generate(**inputs)
        body_raw = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))

        body_tag = "body facing camera"
        if "left" in body_raw:
            body_tag = "body facing left"
        elif "right" in body_raw:
            body_tag = "body facing right"

    # 4. Shot Type
    # User definitions: Full body = feet visible; Upper body = waist visible; Portrait = face close up
    inputs = vqa_processor(image, "Are the person's feet visible in the photo?", return_tensors="pt").to(DEVICE)
    out = vqa_model.generate(**inputs)
    feet_ans = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))

    if "yes" in feet_ans:
        shot_tag = "full body"
    else:
        inputs = vqa_processor(image, "Is the person shown from the waist up?", return_tensors="pt").to(DEVICE)
        out = vqa_model.generate(**inputs)
        waist_ans = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))

        if "yes" in waist_ans:
            shot_tag = "upper body"
        else:
            shot_tag = "portrait"

    # 5. Expression & Mouth
    inputs = vqa_processor(image, "Is the person smiling, frowning, angry, or winking?", return_tensors="pt").to(DEVICE)
    out = vqa_model.generate(**inputs)
    expr_raw = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))

    expr_tag = ""
    if "smile" in expr_raw or "smiling" in expr_raw:
        # Check for both upper and lower teeth
        inputs = vqa_processor(image, "Are the bottom teeth visible?", return_tensors="pt").to(DEVICE)
        out = vqa_model.generate(**inputs)
        bottom_teeth = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))

        inputs = vqa_processor(image, "Are the top teeth visible?", return_tensors="pt").to(DEVICE)
        out = vqa_model.generate(**inputs)
        top_teeth = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))

        if "yes" in bottom_teeth and "yes" in top_teeth:
             expr_tag = "smile, teeth visible"
        else:
             expr_tag = "smile"
    elif "frown" in expr_raw:
        expr_tag = "frown"
    elif "angry" in expr_raw:
        expr_tag = "angry"
    elif "wink" in expr_raw:
        expr_tag = "wink"

    # Mouth state (overrides if distinctive)
    mouth_tag = ""
    inputs = vqa_processor(image, "Is the person puckering their lips?", return_tensors="pt").to(DEVICE)
    out = vqa_model.generate(**inputs)
    special_mouth = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))

    if "pucker" in special_mouth or "yes" in special_mouth:
        mouth_tag = "puckered lips"
    elif not expr_tag: # If neutral expression, check open/closed
        inputs = vqa_processor(image, "Can you see the tongue?", return_tensors="pt").to(DEVICE)
        out = vqa_model.generate(**inputs)
        mouth_state = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))
        if "yes" in mouth_state:
            mouth_tag = "open mouth"
        elif "no" in mouth_state:
            mouth_tag = "closed mouth"

    # 6. Lighting
    inputs = vqa_processor(image, "Is the lighting bright or low light?", return_tensors="pt").to(DEVICE)
    out = vqa_model.generate(**inputs)
    light_raw = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))
    light_tag = "low light" if "low" in light_raw or "dim" in light_raw else ""

    # 7. Accessories
    # Hat
    inputs = vqa_processor(image, "Is the person wearing a hat?", return_tensors="pt").to(DEVICE)
    out = vqa_model.generate(**inputs)
    hat_raw = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))
    hat_tag = "hat" if "yes" in hat_raw else ""

    # Glasses
    inputs = vqa_processor(image, "Is the person wearing glasses?", return_tensors="pt").to(DEVICE)
    out = vqa_model.generate(**inputs)
    glasses_raw = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))
    glasses_tag = "glasses" if "yes" in glasses_raw else ""

    # 8. Hair
    inputs = vqa_processor(image, "Is the hair long?", return_tensors="pt").to(DEVICE)
    out = vqa_model.generate(**inputs)
    long_hair_raw = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))

    if "yes" in long_hair_raw:
        hair_tag = "long hair"
    else:
        hair_tag = "curly hair"

    # 9. Facial Hair
    inputs = vqa_processor(image, "Does the person have a beard or goatee?", return_tensors="pt").to(DEVICE)
    out = vqa_model.generate(**inputs)
    beard_raw = clean_answer(vqa_processor.decode(out[0], skip_special_tokens=True))

    if "yes" in beard_raw:
        facial_hair_tag = "facial hair"
    else:
        facial_hair_tag = "clean shaven"

    # Combine valid tags
    tags = [t for t in [shot_tag, head_tag, looking_tag, body_tag, expr_tag, mouth_tag, light_tag, hat_tag, glasses_tag, hair_tag, facial_hair_tag] if t]
    pose_desc = ", ".join(tags)

    print(f"[Dynamic Analysis] Detected: {pose_desc}")
    return pose_desc

# Test on current input image
if 'input_image' in globals() and input_image is not None:
    print("\nAnalyzing current image...")
    detected_pose = get_dynamic_pose_description(input_image)
else:
    print("\nNo input image loaded to test.")

In [ ]:
# =============================================================================
# RUN SMART INPAINTING (Dynamic Pose + Mask-Aware Shot Type + Zoom)
# =============================================================================
import cv2
import numpy as np

def zoom_inpaint_with_lora(
    base_pipe, refiner_pipe,
    image, mask,
    prompt, negative_prompt,
    padding=64,
    target_size=1024,
    verbose=True,
    **kwargs
):
    """
    Zoom into the masked area, inpaint at high resolution, and paste back.
    Greatly improves quality for small faces.
    """
    if verbose: print(f"🔎 Applying Zoom Inpainting (Target: {target_size}x{target_size})...")

    # 1. Get bounding box of mask
    mask_arr = np.array(mask)
    if len(mask_arr.shape) > 2: mask_arr = mask_arr[:,:,0]
    non_zero = np.where(mask_arr > 128)

    if len(non_zero[0]) == 0:
        if verbose: print("⚠ Empty mask! Fallback to standard inpaint.")
        return inpaint_with_lora(base_pipe, refiner_pipe, image, mask, prompt, negative_prompt, verbose=verbose, **kwargs)

    y_min, y_max = np.min(non_zero[0]), np.max(non_zero[0])
    x_min, x_max = np.min(non_zero[1]), np.max(non_zero[1])

    # 2. Add padding and make square
    h = y_max - y_min
    w = x_max - x_min

    # Center
    cy = (y_min + y_max) // 2
    cx = (x_min + x_max) // 2

    # Size with padding (context)
    context_size = max(h, w) + (padding * 2)
    context_size = max(context_size, 256)

    # Calculate box coordinates
    x1 = cx - context_size // 2
    y1 = cy - context_size // 2
    x2 = x1 + context_size
    y2 = y1 + context_size

    # Adjust for boundaries
    img_w, img_h = image.size
    if x1 < 0: x2 -= x1; x1 = 0
    if y1 < 0: y2 -= y1; y1 = 0
    if x2 > img_w: x1 -= (x2 - img_w); x2 = img_w
    if y2 > img_h: y1 -= (y2 - img_h); y2 = img_h
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(img_w, x2), min(img_h, y2)

    if verbose: print(f"   Zoom Crop Box: ({x1}, {y1}, {x2}, {y2}) | Size: {x2-x1}x{y2-y1}")

    # Crop
    crop_img = image.crop((x1, y1, x2, y2))
    crop_mask = mask.crop((x1, y1, x2, y2))

    # Resize to target
    crop_img_resized = crop_img.resize((target_size, target_size), Image.LANCZOS)
    crop_mask_resized = crop_mask.resize((target_size, target_size), Image.NEAREST)

    # Inpaint
    if verbose: print("   Running inpainting on cropped region...")
    inpainted_resized, seed = inpaint_with_lora(
        base_pipe, refiner_pipe,
        crop_img_resized, crop_mask_resized,
        prompt, negative_prompt,
        verbose=verbose,
        **kwargs
    )

    # Resize back
    inpainted_crop = inpainted_resized.resize((x2-x1, y2-y1), Image.LANCZOS)

    # Paste back
    result = image.copy()
    result.paste(inpainted_crop, (x1, y1), mask=crop_mask)

    return result, seed

# =============================================================================
# CONSOLIDATED LOGIC FUNCTION
# =============================================================================

def smart_inpaint_batch_logic(
    base_pipe,
    refiner_pipe,
    image: Image.Image,
    mask: Image.Image,
    seed: int = -1,
    area_threshold: int = 4000,
    verbose: bool = True
) -> tuple[Image.Image, int]:
    """
    Consolidated logic: Component separation -> Dilation -> VQA -> Inpainting.
    Accepts a full image and a full mask (possibly multiple people).
    """
    if verbose:
        print(f"Running Smart Inpainting Logic...")
        print("-" * 50)

    # 1. Separate people using Connected Components
    mask_np = np.array(mask)
    if len(mask_np.shape) > 2: mask_np = mask_np[:, :, 0] # Ensure single channel
    mask_bin = (mask_np > 128).astype(np.uint8)

    # Find components
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_bin, connectivity=8)

    # Filter and Sort
    components = []
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area > area_threshold: # Filter small areas
            components.append((i, area))

    components.sort(key=lambda x: x[1], reverse=True)
    components = components[:20]

    if verbose: print(f"Found {len(components)} distinct people/regions to inpaint.")

    current_image = image.copy()
    final_seed = seed

    # If no valid components found
    if not components:
         if verbose: print(f"⚠ No components > {area_threshold}px found. Skipping.")
         return current_image, seed

    for idx, (label_id, area) in enumerate(components):
        if verbose: print(f"\n>>> Processing Person {idx+1}/{len(components)} (Area: {area}px)")

        # Extract specific mask
        person_mask_np = (labels == label_id).astype(np.uint8) * 255

        # Expand mask (Dilate) - QUALITY IMPROVEMENT
        dilation_pixels = 15
        kernel = np.ones((dilation_pixels, dilation_pixels), np.uint8)
        person_mask_np = cv2.dilate(person_mask_np, kernel, iterations=1)
        person_mask = Image.fromarray(person_mask_np)

        # Analyze Pose
        y_indices, x_indices = np.where(person_mask_np > 0)
        y_min, y_max = y_indices.min(), y_indices.max()
        x_min, x_max = x_indices.min(), x_indices.max()

        pad = 50
        h, w = current_image.size[1], current_image.size[0]
        crop_y1, crop_y2 = max(0, y_min - pad), min(h, y_max + pad)
        crop_x1, crop_x2 = max(0, x_min - pad), min(w, x_max + pad)

        analysis_crop = current_image.crop((crop_x1, crop_y1, crop_x2, crop_y2))

        if verbose: print("   Analyzing pose...")
        person_pose = get_dynamic_pose_description(analysis_crop)

        # Prompt strategy
        template = PROMPT_HEADSHOT_TEMPLATE
        strength = STRENGTH_HEADSHOT
        guidance = GUIDANCE_HEADSHOT
        smart_prompt = template.format(pose=person_pose)

        smart_negative = NEGATIVE_PROMPT
        if "camera" in person_pose:
             smart_negative = smart_negative.replace("looking at camera, ", "").replace("eye contact, ", "")
        else:
            if "looking at camera" not in smart_negative:
                smart_negative = "looking at camera, eye contact, " + smart_negative

        # Run Inpainting
        current_image, final_seed = zoom_inpaint_with_lora(
            base_pipe=base_pipe,
            refiner_pipe=refiner_pipe,
            image=current_image,
            mask=person_mask,
            prompt=smart_prompt,
            negative_prompt=smart_negative,
            strength=strength,
            guidance_scale=guidance,
            num_inference_steps=NUM_INFERENCE_STEPS,
            high_noise_frac=HIGH_NOISE_FRAC,
            lora_scale=LORA_SCALE,
            seed=seed, # Use passed seed
            padding=100,
            verbose=verbose
        )

    return current_image, final_seed

# =============================================================================
# MAIN EXECUTION LOOP
# =============================================================================

if 'input_image' in globals() and input_image is not None:
    print(f"Running Multi-Person Smart Analysis for @{INPAINT_USERNAME}...")
    print("-" * 50)

    result_image, used_seed = smart_inpaint_batch_logic(
        base_pipe=base_pipe,
        refiner_pipe=refiner_pipe,
        image=input_image,
        mask=mask_image,
        seed=SEED,
        area_threshold=4000,
        verbose=True
    )

    # Display results
    print(f"\nFinal Result (Last seed: {used_seed}):")
    display(make_image_grid([input_image, mask_image, result_image], rows=1, cols=3))

    # Save
    output_path = f"/content/{INPAINT_USERNAME}_smart_inpainted.png"
    result_image.save(output_path)
    print(f"✓ Saved to {output_path}")
else:
    print("❌ No input image. Please run the 'Fetch Image' cell first.")

## 9. Save Result

In [ ]:
# Save the result
if 'result_image' in dir() and result_image is not None:
    output_path = f"/content/{INPAINT_USERNAME}_inpainted_seed{used_seed}.png"
    result_image.save(output_path, "PNG")
    print(f"✓ Saved to: {output_path}")

    # Download the result
    try:
        from google.colab import files
        files.download(output_path)
    except ImportError:
        print("(Download not available in local environment)")
else:
    print("❌ No result to save. Run inpainting first.")

## 10. Batch Processing (Optional)

Process multiple profiles from R2 with automatic mask generation.

## 10a. Metadata Migration Script

Run this cell once to update existing profiles_metadata.json with new fields for v1 processing.
This adds `r2_original_upload_status`, `r2_original_error`, `has_people`, `v1_image_r2_key`, and `v1_error` fields.

In [ ]:
# =============================================================================
# METADATA MIGRATION SCRIPT
# Run this once to add new fields to existing profiles_metadata.json
# =============================================================================

def migrate_metadata_schema(metadata_path: str, dry_run: bool = True) -> dict:
    """
    Migrate existing profiles_metadata.json to new schema with v1 processing fields.

    New fields added:
    - r2_original_upload_status: renamed from r2_upload_status
    - r2_original_error: renamed from r2_error
    - has_people: null (to be detected during processing)
    - v1_image_r2_key: null (R2 key for processed v1 image)
    - v1_error: null (error during v1 processing if any)
    - v1_processed_at: null (timestamp of v1 processing)
    - v2_image_r2_key: null (R2 key for processed v2 image)
    - v2_error: null (error during v2 processing if any)
    - v2_processed_at: null (timestamp of v2 processing)

    Args:
        metadata_path: Path to profiles_metadata.json
        dry_run: If True, don't save changes, just show what would happen

    Returns:
        Migrated metadata dict
    """
    import json
    import shutil
    from datetime import datetime

    print(f"Loading metadata from: {metadata_path}")
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)

    profiles = metadata.get('profiles', {})
    total = len(profiles)
    migrated = 0
    already_migrated = 0

    print(f"Found {total} profiles to check...")

    for profile_id, profile in profiles.items():
        needs_migration = False

        # Rename r2_upload_status -> r2_original_upload_status (if old field exists)
        if 'r2_upload_status' in profile and 'r2_original_upload_status' not in profile:
            profile['r2_original_upload_status'] = profile.pop('r2_upload_status')
            needs_migration = True
        elif 'r2_original_upload_status' not in profile:
            profile['r2_original_upload_status'] = None
            needs_migration = True

        # Rename r2_error -> r2_original_error (if old field exists)
        if 'r2_error' in profile and 'r2_original_error' not in profile:
            profile['r2_original_error'] = profile.pop('r2_error')
            needs_migration = True
        elif 'r2_original_error' not in profile:
            profile['r2_original_error'] = None
            needs_migration = True

        # Add new v1 and v2 processing fields if not present
        new_fields = {
            'has_people': None,
            'v1_image_r2_key': None,
            'v1_error': None,
            'v1_processed_at': None,
            'v2_image_r2_key': None,
            'v2_error': None,
            'v2_processed_at': None
        }

        for field, default_value in new_fields.items():
            if field not in profile:
                profile[field] = default_value
                needs_migration = True

        if needs_migration:
            migrated += 1
        else:
            already_migrated += 1

    print(f"\n{'='*50}")
    print(f"Migration Summary:")
    print(f"  - Total profiles: {total}")
    print(f"  - Profiles migrated: {migrated}")
    print(f"  - Already up-to-date: {already_migrated}")
    print(f"{'='*50}")

    if not dry_run and migrated > 0:
        # Create backup
        backup_path = metadata_path + f".bak.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        shutil.copy2(metadata_path, backup_path)
        print(f"\n✓ Backup created: {backup_path}")

        # Save migrated metadata
        metadata['last_updated'] = datetime.now().isoformat()
        with open(metadata_path, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)
        print(f"✓ Metadata saved: {metadata_path}")
    elif dry_run:
        print("\n[DRY RUN] No changes saved. Set dry_run=False to apply changes.")
    else:
        print("\n✓ No migration needed - all profiles already have new schema.")

    return metadata

# =============================================================================
# Run migration (set dry_run=False to actually save changes)
# =============================================================================
METADATA_PATH = "/content/drive/MyDrive/Loras/wesleygram/profiles_metadata.json"  # @param {type:"string"}
DRY_RUN = False  # @param {type:"boolean"}

# Mount Google Drive if in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted")
except ImportError:
    print("Not in Colab - using local path")

# Run migration
if os.path.exists(METADATA_PATH):
    migrated_metadata = migrate_metadata_schema(METADATA_PATH, dry_run=DRY_RUN)
else:
    print(f"❌ Metadata file not found: {METADATA_PATH}")
    print("Please check the path and mount Google Drive if needed.")

## 10b. Batch V1 Processing from R2

Process followers/following profiles in batches:
1. Load metadata from Google Drive
2. Filter profiles by mode (followers, following, both)
3. Download original image from R2
4. Detect if person/people are present using ML
5. If no people detected → set `has_people=False` and skip inference
6. If people detected → run inpainting pipeline with dynamic pose detection
7. Upload processed image to R2 as `v1/filename`
8. Update metadata and save back to Drive

**Memory Management**: Segmentation models are freed before each inference run to conserve GPU VRAM.

In [ ]:
# @title 🚀 Batch V2 Processing Pipeline
# @markdown ### Configuration
# @markdown Configure the batch processing parameters below:

# =============================================================================
# BATCH V2 PROCESSING CONFIGURATION (Colab Form Inputs)
# =============================================================================

# @markdown ---
# @markdown #### Source & Target Settings
METADATA_PATH = "/content/drive/MyDrive/Loras/wesleygram/profiles_metadata.json"  # @param {type:"string"}
PROCESSING_MODE = "followers"  # @param ["followers", "following", "both"]
BATCH_LIMIT = 1  # @param {type:"integer"}

# @markdown ---
# @markdown #### Processing Options
SKIP_ALREADY_PROCESSED = True  # @param {type:"boolean"}
SAVE_AFTER_EACH = True  # @param {type:"boolean"}
UPLOAD_TO_R2 = True  # @param {type:"boolean"}
VERBOSE = False # @param {type:"boolean"}

# @markdown ---
# @markdown #### Advanced Options (expand to modify)
DETECTION_THRESHOLD = 0.3  # @param {type:"number"}
MASK_EXPANSION = 10  # @param {type:"integer"}
MASK_FEATHER = 5  # @param {type:"integer"}

# =============================================================================
# BATCH PROCESSING IMPLEMENTATION
# =============================================================================

import json
import gc
import torch
import time
import cv2
import numpy as np
import boto3
from datetime import datetime
from io import BytesIO
from pathlib import Path
from enum import Enum
from PIL import Image, ImageFilter

class ProcessingMode(Enum):
    FOLLOWERS = "followers"
    FOLLOWING = "following"
    BOTH = "both"

def load_metadata_from_drive(metadata_path: str) -> dict:
    """Load profiles metadata from Google Drive."""
    print(f"Loading metadata from: {metadata_path}")
    with open(metadata_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_metadata_to_drive(metadata: dict, metadata_path: str):
    """Save profiles metadata back to Google Drive with atomic write."""
    import shutil

    tmp_path = metadata_path + ".tmp"
    bak_path = metadata_path + ".bak"

    metadata['last_updated'] = datetime.now().isoformat()

    # Write to temp file
    with open(tmp_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    # Backup existing
    if os.path.exists(metadata_path):
        shutil.copy2(metadata_path, bak_path)

    # Atomic replace
    os.replace(tmp_path, metadata_path)
    # print(f"✓ Metadata saved to: {metadata_path}") # Reduce spam

def filter_profiles_by_mode(
    profiles: dict,
    mode: ProcessingMode,
    skip_processed: bool = True
) -> list[tuple[str, dict]]:
    """
    Filter profiles based on processing mode and skip already processed (V2).
    """
    candidates = []

    for profile_id, profile in profiles.items():
        # Check mode filter
        is_follower = profile.get('is_follower', False)
        is_following = profile.get('is_following', False)

        if mode == ProcessingMode.FOLLOWERS and not is_follower:
            continue
        elif mode == ProcessingMode.FOLLOWING and not is_following:
            continue
        elif mode == ProcessingMode.BOTH and not (is_follower or is_following):
            continue

        # Check if already processed (V2 Check)
        if skip_processed:
            if profile.get('v2_image_r2_key') is not None:
                continue
            # Also skip if explicitly marked as no people
            if profile.get('has_people') == False:
                continue

        # Check if has original image in R2
        if not profile.get('original_image_r2_key'):
            continue

        candidates.append((profile_id, profile))

    return candidates

def upload_v2_to_r2(
    s3_client,
    bucket: str,
    image: Image.Image,
    instagram_id: str
) -> tuple[str, str]:
    """
    Upload processed v2 image to R2.
    """
    try:
        # Generate filename with v2 prefix
        timestamp = int(time.time())
        r2_key = f"v2/{instagram_id}_{timestamp}.png"

        # Convert image to bytes
        buffer = BytesIO()
        image.save(buffer, format='PNG', optimize=True)
        buffer.seek(0)

        # Upload to R2
        s3_client.put_object(
            Bucket=bucket,
            Key=r2_key,
            Body=buffer.getvalue(),
            ContentType='image/png'
        )

        return r2_key, None

    except Exception as e:
        error_msg = str(e)
        print(f"✗ R2 upload failed: {error_msg}")
        return None, error_msg

def cleanup_segmenter(segmenter):
    """Free segmentation model memory."""
    if segmenter is not None:
        try:
            segmenter.cleanup()
        except:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def batch_v2_process(
    metadata_path: str,
    mode: str,
    limit: int,
    r2_creds: dict,
    skip_processed: bool = True,
    save_after_each: bool = True,
    upload_to_r2: bool = True,
    detection_threshold: float = 0.3,
    mask_expansion: int = 10,
    mask_feather: int = 5,
    verbose: bool = True
) -> dict:
    """
    Main batch processing loop for v2 inference.
    """
    processing_mode = ProcessingMode(mode)

    # Load metadata
    metadata = load_metadata_from_drive(metadata_path)
    profiles = metadata.get('profiles', {})

    # Filter profiles
    candidates = filter_profiles_by_mode(profiles, processing_mode, skip_processed)
    total_candidates = len(candidates)

    print(f"\n{'='*60}")
    print(f"🚀 BATCH V2 PROCESSING")
    print(f"{'='*60}")
    print(f"  Mode: {mode}")
    print(f"  Total candidates: {total_candidates}")
    print(f"  Processing limit: {limit}")
    print(f"  Verbose: {verbose}")
    print(f"{'='*60}\n")

    if total_candidates == 0:
        print("✓ No profiles to process!")
        return {'processed': 0, 'skipped': 0, 'errors': 0}

    # Limit to batch size
    to_process = candidates[:limit] if limit else candidates

    # Initialize R2 client
    s3_client = boto3.client(
        's3',
        endpoint_url=r2_creds['r2_endpoint'],
        aws_access_key_id=r2_creds['r2_access_key'],
        aws_secret_access_key=r2_creds['r2_secret_key']
    )
    bucket = r2_creds['r2_bucket']

    # Stats
    stats = {
        'processed': 0,
        'no_people': 0,
        'errors': 0,
        'error_details': []
    }

    for idx, (profile_id, profile) in enumerate(to_process):
        username = profile.get('username', profile_id)
        r2_key = profile.get('original_image_r2_key')

        # Header
        if verbose:
            print(f"\n{'='*60}")
            print(f"[{idx+1}/{len(to_process)}] Processing: @{username} (ID: {profile_id})")
            print(f"{'='*60}")
        else:
            print(f"[{idx+1}/{len(to_process)}] @{username}", end=" ")

        try:
            # Step 1: Download image from R2
            if verbose: print(f"  📥 Downloading from R2: {r2_key}")
            response = s3_client.get_object(Bucket=bucket, Key=r2_key)
            image_bytes = response['Body'].read()
            raw_image = Image.open(BytesIO(image_bytes)).convert('RGB')
            if verbose: print(f"  ✓ Downloaded: {raw_image.size[0]}x{raw_image.size[1]}")

            # Step 2: Initialize segmenter and Generate FULL Mask
            # This replaces the previous box-loop with the single-loop strategy
            if verbose: print(f"  🔍 Generating person mask...")
            segmenter = PersonSegmenter(
                sam_checkpoint=SAM_CHECKPOINT,
                sam_model_type=SAM_MODEL_TYPE,
                detection_model=DETECTION_MODEL,
                device=DEVICE,
                verbose=verbose
            )

            # Use generate_mask to get the full mask (multiscale detection)
            full_mask_pil = segmenter.generate_mask(
                raw_image,
                detection_threshold=detection_threshold,
                expansion_pixels=mask_expansion,
                feather_radius=mask_feather
            )

            # Check if mask is empty
            has_people = np.any(np.array(full_mask_pil) > 0)

            # Update has_people field
            profile['has_people'] = bool(has_people)

            if not has_people:
                if verbose: print(f"  ⚠ No people detected (mask empty) - skipping.")
                else: print("→ No people detected.")

                profile['v2_error'] = "no_people_detected"
                profile['v2_processed_at'] = datetime.now().isoformat()
                stats['no_people'] += 1
                cleanup_segmenter(segmenter)

                if save_after_each:
                    save_metadata_to_drive(metadata, metadata_path)
                continue

            if verbose: print(f"  ✓ Mask generated.")

            # Step 3: Preprocess (Resize/Crop) to SDXL Target Resolution
            target_size = RESOLUTION
            aspect = raw_image.width / raw_image.height

            if aspect > 1:
                new_width = int(target_size * aspect)
                new_height = target_size
            else:
                new_width = target_size
                new_height = int(target_size / aspect)

            left = (new_width - target_size) // 2
            top = (new_height - target_size) // 2

            # Resize Input Image
            input_image = raw_image.resize((new_width, new_height), Image.Resampling.LANCZOS)
            input_image = input_image.crop((left, top, left + target_size, top + target_size))

            # Resize Mask
            mask_image = full_mask_pil.resize((new_width, new_height), Image.Resampling.NEAREST)
            mask_image = mask_image.crop((left, top, left + target_size, top + target_size))

            # Step 4: Free segmenter memory before inference
            if verbose: print(f"  🧹 Freeing segmentation memory for inference...")
            cleanup_segmenter(segmenter)
            del segmenter
            segmenter = None

            # Step 5: Run CONSOLIDATED Smart Inpainting Logic
            # This runs connected components, dilation, VQA, and zoom inpainting
            if verbose: print(f"  🎨 Running Smart Inpainting Logic...")
            else: print("→ Processing...", end=" ")

            result_image, final_seed = smart_inpaint_batch_logic(
                base_pipe=base_pipe,
                refiner_pipe=refiner_pipe,
                image=input_image,
                mask=mask_image,
                seed=SEED,
                area_threshold=4000,
                verbose=verbose
            )

            if verbose:
                print(f"  ✓ Processing complete (seed: {final_seed})")
            else:
                print("✓ Done.", end=" ")

            # Step 6: Upload to R2 (V2 prefix)
            if upload_to_r2:
                if verbose: print(f"  📤 Uploading to R2...")
                v2_key, v2_error = upload_v2_to_r2(
                    s3_client, bucket, result_image, profile_id
                )
                profile['v2_image_r2_key'] = v2_key
                profile['v2_error'] = v2_error

                if not verbose and v2_key:
                    print(f"✓ Uploaded: {v2_key}")
                elif not verbose:
                     print(f"✗ Upload failed: {v2_error}")
            else:
                if verbose: print(f"  ⏭ Skipping R2 upload (disabled)")
                profile['v2_image_r2_key'] = None
                profile['v2_error'] = "upload_disabled"

            profile['v2_processed_at'] = datetime.now().isoformat()
            stats['processed'] += 1

            if verbose: print(f"  ✓ Successfully processed @{username}")

            # Display result (verbose only)
            if verbose:
                print(f"\n  Result preview:")
                display(make_image_grid([input_image, mask_image, result_image], rows=1, cols=3))

        except KeyboardInterrupt:
            print(f"\n\n🛑 Batch processing stopped by user at @{username}.")
            break

        except Exception as e:
            error_msg = str(e)
            if verbose: print(f"  ✗ ERROR: {error_msg}")
            else: print(f"✗ ERROR: {error_msg}")

            profile['v2_error'] = error_msg
            profile['v2_processed_at'] = datetime.now().isoformat()
            stats['errors'] += 1
            stats['error_details'].append({'username': username, 'error': error_msg})

            # Cleanup on error
            if 'segmenter' in dir() and segmenter is not None:
                cleanup_segmenter(segmenter)

        finally:
            # Save after each if enabled
            if save_after_each:
                save_metadata_to_drive(metadata, metadata_path)

            # Clear CUDA cache between profiles
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # Final save
    if not save_after_each:
        save_metadata_to_drive(metadata, metadata_path)

    # Print summary
    print(f"\n{'='*60}")
    print(f"🏁 BATCH PROCESSING COMPLETE (V2)")
    print(f"{'='*60}")
    print(f"  ✓ Successfully processed: {stats['processed']}")
    print(f"  ⚠ No people (skipped): {stats['no_people']}")
    print(f"  ✗ Errors: {stats['errors']}")
    if stats['error_details']:
        print(f"\n  Error details:")
        for err in stats['error_details']:
            print(f"    - @{err['username']}: {err['error'][:50]}...")
    print(f"{'='*60}")

    return stats

# =============================================================================
# RUN BATCH PROCESSING
# =============================================================================

# Mount Google Drive if needed
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
        print("✓ Google Drive mounted")
except ImportError:
    print("Not in Colab environment")

# Verify metadata exists
if not os.path.exists(METADATA_PATH):
    print(f"❌ Metadata file not found: {METADATA_PATH}")
    print("Please verify the path and ensure Google Drive is mounted.")
else:
    print(f"✓ Metadata file found: {METADATA_PATH}")

    # Check if pipelines are loaded
    if 'base_pipe' not in dir() or base_pipe is None:
        print("\n⚠ SDXL pipelines not loaded!")
        print("Please run the 'Load Models' cell (Section 5) first.")
    elif 'get_dynamic_pose_description' not in dir():
        print("\n⚠ BLIP-VQA not loaded!")
        print("Please run the 'Dynamic Pose Detection' cell first.")
    else:
        print("\n✓ All dependencies ready. Starting batch V2 processing...")

        # Run batch processing
        results = batch_v2_process(
            metadata_path=METADATA_PATH,
            mode=PROCESSING_MODE,
            limit=BATCH_LIMIT,
            r2_creds=r2_creds,
            skip_processed=SKIP_ALREADY_PROCESSED,
            save_after_each=SAVE_AFTER_EACH,
            upload_to_r2=UPLOAD_TO_R2,
            detection_threshold=DETECTION_THRESHOLD,
            mask_expansion=MASK_EXPANSION,
            mask_feather=MASK_FEATHER,
            verbose=VERBOSE
        )

In [ ]:
def batch_inpaint_from_r2(
    r2_fetcher,
    base_pipe,
    refiner_pipe,
    usernames: list[str],
    output_dir: str,
    prompt: str,
    negative_prompt: str,
    sam_checkpoint: str = SAM_CHECKPOINT,
    sam_model_type: str = SAM_MODEL_TYPE,
    detection_model: str = DETECTION_MODEL,
    **kwargs
) -> list[dict]:
    """
    Batch process multiple profiles from R2 with automatic person segmentation.

    Args:
        r2_fetcher: R2ProfileFetcher instance
        base_pipe: SDXL base inpainting pipeline
        refiner_pipe: SDXL refiner pipeline
        usernames: List of Instagram usernames to process
        output_dir: Directory to save results
        prompt: Generation prompt
        negative_prompt: Negative prompt
        **kwargs: Additional args for inpaint_with_lora

    Returns:
        List of result dicts with username, paths, and status
    """
    os.makedirs(output_dir, exist_ok=True)

    # Load segmentation models for batch
    print("Loading segmentation models for batch processing...")
    batch_segmenter = PersonSegmenter(
        sam_checkpoint=sam_checkpoint,
        sam_model_type=sam_model_type,
        detection_model=detection_model,
        device=kwargs.get('device', 'cuda')
    )

    results = []

    for i, username in enumerate(usernames):
        print(f"\n{'='*60}")
        print(f"Processing {i+1}/{len(usernames)}: @{username}")
        print(f"{'='*60}")

        result_entry = {
            'username': username,
            'status': 'pending',
            'input_path': None,
            'mask_path': None,
            'output_path': None,
            'error': None
        }

        try:
            # Fetch image from R2
            raw_image = r2_fetcher.fetch_image(username)

            # Generate mask
            mask = batch_segmenter.generate_mask(
                raw_image,
                detection_threshold=kwargs.get('detection_threshold', 0.3),
                expansion_pixels=kwargs.get('expansion_pixels', 10),
                feather_radius=kwargs.get('feather_radius', 5)
            )

            # Preprocess
            image, mask = preprocess_image_and_mask_from_pil(
                image=raw_image,
                mask=mask,
                target_size=kwargs.get('resolution', 1024)
            )

            # Save input and mask
            input_path = f"{output_dir}/{username}_input.png"
            mask_path = f"{output_dir}/{username}_mask.png"
            image.save(input_path)
            mask.save(mask_path)
            result_entry['input_path'] = input_path
            result_entry['mask_path'] = mask_path

            # Inpaint
            inpainted, seed = inpaint_with_lora(
                base_pipe, refiner_pipe,
                image, mask,
                prompt, negative_prompt,
                **kwargs
            )

            # Save result
            output_path = f"{output_dir}/{username}_inpainted_seed{seed}.png"
            inpainted.save(output_path)
            result_entry['output_path'] = output_path
            result_entry['status'] = 'success'
            result_entry['seed'] = seed

            print(f"✓ Saved: {output_path}")

        except Exception as e:
            result_entry['status'] = 'failed'
            result_entry['error'] = str(e)
            print(f"❌ Failed: {e}")

        results.append(result_entry)

        # Clear cache between images
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Cleanup segmentation models
    batch_segmenter.cleanup()
    del batch_segmenter
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Summary
    success = sum(1 for r in results if r['status'] == 'success')
    failed = sum(1 for r in results if r['status'] == 'failed')

    print(f"\n{'='*60}")
    print(f"Batch complete!")
    print(f"  ✓ Success: {success}")
    print(f"  ✗ Failed: {failed}")
    print(f"{'='*60}")

    return results

In [ ]:
# =============================================================================
# Example: Batch process multiple usernames from R2
# =============================================================================

# List of usernames to process
BATCH_USERNAMES = [
    "0xkaii",
    "15harshit",
    # Add more usernames here
]

# Uncomment to run batch processing:
# batch_results = batch_inpaint_from_r2(
#     r2_fetcher=r2_fetcher,
#     base_pipe=base_pipe,
#     refiner_pipe=refiner_pipe,
#     usernames=BATCH_USERNAMES,
#     output_dir="/content/batch_outputs",
#     prompt=PROMPT,
#     negative_prompt=NEGATIVE_PROMPT,
#     num_inference_steps=NUM_INFERENCE_STEPS,
#     guidance_scale=GUIDANCE_SCALE,
#     high_noise_frac=HIGH_NOISE_FRAC,
#     lora_scale=LORA_SCALE,
#     seed=-1,  # Random seed for each
#     device=DEVICE,
#     detection_threshold=PERSON_DETECTION_THRESHOLD,
#     expansion_pixels=MASK_EXPANSION_PIXELS,
#     feather_radius=MASK_FEATHER_RADIUS
# )

# Download all results as zip:
# import shutil
# shutil.make_archive("/content/batch_results", "zip", "/content/batch_outputs")
# files.download("/content/batch_results.zip")

## 11. Experiment with Different Settings

In [ ]:
# Interactive experimentation - modify these and re-run
EXPERIMENT_PROMPT = "photo of wesleykamau person male, realistic face, natural skin texture, matching lighting, photorealistic"
EXPERIMENT_NEG = "blurry, distorted face, bad anatomy, extra eyes, unrealistic skin"
EXPERIMENT_STEPS = 30
EXPERIMENT_GUIDANCE = 7.5
EXPERIMENT_LORA_SCALE = 0.9
EXPERIMENT_SEED = -1  # Random

if 'input_image' in dir() and 'mask_image' in dir():
    exp_result, exp_seed = inpaint_with_lora(
        base_pipe, refiner_pipe,
        input_image, mask_image,
        EXPERIMENT_PROMPT, EXPERIMENT_NEG,
        num_inference_steps=EXPERIMENT_STEPS,
        guidance_scale=EXPERIMENT_GUIDANCE,
        high_noise_frac=0.8,
        lora_scale=EXPERIMENT_LORA_SCALE,
        seed=EXPERIMENT_SEED,
        device=DEVICE
    )

    print(f"\nExperiment result (seed: {exp_seed}):")
    display(exp_result)

## 12. Memory Cleanup

In [ ]:
# Clean up GPU memory when done
def cleanup():
    global base_pipe, refiner_pipe

    if 'base_pipe' in dir():
        del base_pipe
    if 'refiner_pipe' in dir():
        del refiner_pipe

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

    print("Cleanup complete!")

# Uncomment to run cleanup
# cleanup()

---

## Tips for Best Results

### R2 Integration
- **Colab secrets**: Add `R2_ENDPOINT_URL`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY` in Settings → Secrets
- **Local development**: Create a `.env` file with the same variables
- **Metadata**: The `profiles_metadata.json` maps usernames to R2 keys

### Automatic Person Segmentation
- **Detection threshold**: Lower (0.2) = more sensitive, may include false positives. Higher (0.4) = stricter
- **Expansion pixels**: Increase (15-20) if masks are too tight around the person
- **Feather radius**: Higher values (7-10) create softer edges for better blending

### Inpainting
1. **Prompt Engineering**: Always start with your trigger token (`wesleykamau person male`)
2. **LoRA Scale**: Start at 0.8-0.9, reduce if results are too stylized
3. **Guidance Scale**: 7-8 works well for realistic results
4. **Steps**: 25-30 provides good quality without excessive compute
5. **Seed**: Use fixed seeds to iterate on prompts, random for variety

### Models Used
- **Grounding DINO**: Zero-shot object detection for finding people in images
- **SAM (Segment Anything)**: Precise segmentation masks from bounding boxes
- **SDXL Base**: High-quality image generation
- **SDXL Refiner**: Detail enhancement using ensemble of expert denoisers

### Memory Optimization
- Segmentation models are unloaded after mask generation to free VRAM for SDXL
- Base and Refiner share VAE and text_encoder_2 to reduce memory usage
- Use `enable_model_cpu_offload()` if running out of VRAM

### Testing Section
Use the 🧪 Testing Section cells to quickly test mask generation on different usernames
without running the full SDXL pipeline.